In [27]:
from typing import Optional

In [2]:
class Problema_Baldes:

# OPERAÇÕES INTERNAS
    def __init__(self, baldes: Optional[list[int]] = None):
        self.baldes: list[int] = baldes if baldes is not None else [5, 0]

    def _get_b5(self) -> int:
        return self.baldes[0]

    def _get_b3(self) -> int:
        return self.baldes[1]

    def _set_b5(self, valor: int) -> bool:
        if valor > 5 or valor < 0:
            return False
        self.baldes[0] = valor
        return True

    def _set_b3(self, valor: int) -> bool:
        if valor > 3 or valor < 0:
            return False
        self.baldes[1] = valor
        return True

# OPERAÇÕES AUXILIARES
    def clonar(self) -> "Problema_Baldes":
        return Problema_Baldes(self.baldes.copy())

    def tupla(self) -> tuple[int, int]:
        return (self.baldes[0], self.baldes[1])

    def print(self):
        print(f'BALDE MAIOR: {self.baldes[0]}')
        print(f'BALDE MENOR: {self.baldes[1]}')

    def is_solucionado(self) -> bool:
        return (self._get_b3() + self._get_b5()) == 4

# REGRAS DE TRANSIÇÃO
    def enche_b5(self) -> bool:
        valor = self._get_b5()
        if valor >= 5:
            return False
        self._set_b5(5)
        return True

    def enche_b3(self) -> bool:
        valor = self._get_b3()
        if valor >= 3:
            return False
        self._set_b3(3)
        return True

    def esvazia_b5(self) -> bool:
        valor = self._get_b5()
        if valor < 1:
            return False
        self._set_b5(0)
        return True

    def esvazia_b3(self) -> bool:
        valor = self._get_b3()
        if valor < 1:
            return False
        self._set_b3(0)
        return True

    def b5_to_b3(self) -> bool:
        valor_maior = self._get_b5()
        valor_menor = self._get_b3()

        if valor_maior == 0 or valor_menor >= 3:
            return False

        folga_menor = 3 - valor_menor
        if valor_maior <= folga_menor:
            self._set_b3(valor_menor + valor_maior)
            self._set_b5(0)
        else:
            self._set_b3(3)
            self._set_b5(valor_maior - folga_menor)
        return True

    def b3_to_b5(self) -> bool:
        valor_maior = self._get_b5()
        valor_menor = self._get_b3()

        if valor_menor == 0 or valor_maior >= 5:
            return False

        folga_maior = 5 - valor_maior
        if valor_menor <= folga_maior:
            self._set_b5(valor_menor + valor_maior)
            self._set_b3(0)
        else:
            self._set_b5(5)
            self._set_b3(valor_menor - folga_maior)
        return True

NameError: name 'Optional' is not defined

In [29]:
class Estado_Busca:

    ORDEM_EXECUCAO = [
        "regra_5",
        "regra_2",
        "regra_4",
        "regra_3",
        "regra_1",
        "regra_6",
    ]
    
    def __init__(self, pai: Optional["Estado_Busca"] = None, baldes: Problema_Baldes = None):
        self.baldes = baldes if baldes else Problema_Baldes()
        self.pai = pai
        self.proxima_regra = 0
        self.impasse = False
        self.regras = {
            "regra_1": self.baldes.enche_b3,
            "regra_2": self.baldes.enche_b5,
            "regra_3": self.baldes.esvazia_b3,
            "regra_4": self.baldes.esvazia_b5,
            "regra_5": self.baldes.b3_to_b5,
            "regra_6": self.baldes.b5_to_b3,
        }
        

    def gerar_filho(self):
        while self.proxima_regra < len(self.ORDEM_EXECUCAO):
            nome_regra = self.ORDEM_EXECUCAO[self.proxima_regra]
            self.proxima_regra += 1
            
            candidato = Estado_Busca(pai=self, baldes=self.baldes.clonar())
            if candidato.regras[nome_regra]():
                return candidato
        self.impasse = True
        return None

In [2]:
class Busca_Baldes:
    def __init__(self):
        self.baldes = Problema_Baldes()
        self.nivel = 0
        self.estados = [Estado_Busca(pai=None, baldes=self.baldes)]
        self.visitados = {self.baldes.tupla()}

    def print(self):
        self.estados[self.nivel].baldes.print()

    def busca_completa(self) -> bool:
        while not self.estados[self.nivel].baldes.is_solucionado():
            estado_atual = self.estados[self.nivel]
            novo_estado = estado_atual.gerar_filho()

            if novo_estado is not None:
                chave = (novo_estado.baldes.tupla())
                if chave not in self.visitados:
                    self.visitados.add(chave)
                    self.nivel += 1
                    self.estados.append(novo_estado)
            else:
                if self.nivel == 0:
                    return False
                self.estados.pop()
                self.nivel -= 1

        return True

    def imprime_solucao(self):
        caminho = []
        estado = self.estados[self.nivel]
        while estado is not None:
            caminho.append(estado)
            estado = estado.pai
        caminho.reverse()

        for i, estado in enumerate(caminho):
            print(f'--- Passo {i} ---')
            estado.baldes.print()

In [1]:
if __name__ == "__main__":
    busca = Busca_Baldes()
    if busca.busca_completa():
        print("Solução encontrada!\n")
        busca.imprime_solucao()
    else:
        print("Não foi possível encontrar solução (impasse).")

NameError: name 'Busca_Baldes' is not defined